In [1]:
# track.py
class TrackSegment:
    def __init__(self, ds, curvature):
        self.ds = ds
        self.curvature = curvature
# vehicle.py
class Vehicle:
    def __init__(self):
        self.mu = 1.2
        self.g = 9.81
        self.accel = 4.0
        self.brake = 8.0


In [2]:
# physics.py
import math

def lateral_speed_limit(mu, g, curvature):
    if curvature == 0:
        return float("inf")
    return math.sqrt(mu * g / curvature)


In [3]:
# sim.py
def simulate(track, vehicle):
    n = len(track)
    v = [0.0] * n

    # forward pass
    for i in range(n - 1):
        v_next = math.sqrt(v[i]**2 + 2 * vehicle.accel * track[i].ds)
        v_lat = lateral_speed_limit(vehicle.mu, vehicle.g, track[i+1].curvature)
        v[i+1] = min(v_next, v_lat)

    # backward pass
    for i in reversed(range(n - 1)):
        v_allowed = math.sqrt(v[i+1]**2 + 2 * vehicle.brake * track[i].ds)
        v[i] = min(v[i], v_allowed)

    # lap time
    lap_time = sum(track[i].ds / v[i] for i in range(1, n))
    return lap_time, v


In [6]:
import math

# -----------------------------
# Track definition
# -----------------------------

class TrackSegment:
    def __init__(self, ds, curvature):
        self.ds = ds              # segment length (m)
        self.curvature = curvature  # 1/radius (1/m)


def build_track():
    """
    Simple track:
    - Straight
    - Straight
    - Corner
    - Corner
    - Straight
    """
    ds = 10.0  # meters per segment

    return [
        TrackSegment(ds, 0.0),
        TrackSegment(ds, 0.0),
        TrackSegment(ds, 0.10),
        TrackSegment(ds, 0.10),
        TrackSegment(ds, 0.0),
    ]


# -----------------------------
# Vehicle definition
# -----------------------------

class Vehicle:
    def __init__(self):
        self.mu = 1.2          # tire friction coefficient
        self.g = 9.81          # gravity (m/s^2)
        self.accel = 4.0       # acceleration (m/s^2)
        self.brake = 8.0       # braking (m/s^2)


# -----------------------------
# Physics functions
# -----------------------------

def lateral_speed_limit(mu, g, curvature):
    if curvature == 0.0:
        return float("inf")
    return math.sqrt(mu * g / curvature)


# -----------------------------
# Lap time simulation
# -----------------------------

def simulate_lap(track, vehicle):
    n = len(track)
    v = [0.0] * n  # speed at each segment start

    # ---- Forward pass (acceleration) ----
    for i in range(n - 1):
        ds = track[i].ds
        v_possible = math.sqrt(v[i] ** 2 + 2 * vehicle.accel * ds)

        v_lat = lateral_speed_limit(
            vehicle.mu, vehicle.g, track[i + 1].curvature
        )

        v[i + 1] = min(v_possible, v_lat)

    # ---- Backward pass (braking) ----
    for i in reversed(range(n - 1)):
        ds = track[i].ds
        v_allowed = math.sqrt(v[i + 1] ** 2 + 2 * vehicle.brake * ds)
        v[i] = min(v[i], v_allowed)

    # ---- Lap time integration ----
    lap_time = 0.0
    for i in range(1, n):
        lap_time += track[i - 1].ds / v[i]

    return lap_time, v

import csv

def load_track_from_csv(path):
    track = []
    with open(path, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            track.append(
                TrackSegment(
                    ds=float(row["ds"]),
                    curvature=float(row["curvature"])
                )
            )
    return track

# -----------------------------
# Main
# -----------------------------

if __name__ == "__main__":
    track = load_track_from_csv("tracks/silverstone.csv")
    # track = build_track()
    vehicle = Vehicle()

    lap_time, speeds = simulate_lap(track, vehicle)

    print("Segment speeds (m/s):")
    for i, v in enumerate(speeds):
        print(f" Segment {i}: {v:.2f}")

    print(f"\nTotal lap time: {lap_time:.2f} seconds")


Segment speeds (m/s):

Total lap time: 0.00 seconds
